# 02 - Modelagem (Regressao)

**FarmTech Solutions / Fase 4 CAP 1**

Treina e compara modelos de regressao para prever `produtividade_kg_ha`.
Usa o pre-processador canonico de `feature_engineering` (scaling + one-hot da
cultura) dentro de um `Pipeline`, garantindo o mesmo tratamento no treino e na
predicao. Comparacao por validacao cruzada de 5 folds; o melhor modelo e
salvo em `models/`.

In [1]:
import sys
from pathlib import Path
DATA_DIR = Path.cwd().parent / 'data'
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))
import feature_engineering as fe

import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42
MODELS_DIR = Path.cwd() / 'models'
MODELS_DIR.mkdir(exist_ok=True)

df = fe.carregar_dados()
X, y = fe.separar_xy(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Treino:', X_train.shape, '| Teste:', X_test.shape)

Treino: (1760, 9) | Teste: (440, 9)


## Definicao dos modelos

Cada modelo e um `Pipeline`: pre-processador + estimador.

In [2]:
def montar(estimador, poly=False):
    etapas = [('prep', fe.construir_preprocessador())]
    if poly:
        etapas.append(('poly', PolynomialFeatures(degree=2, include_bias=False)))
    etapas.append(('model', estimador))
    return Pipeline(etapas)

modelos = {
    'LinearRegression': montar(LinearRegression()),
    'Ridge': montar(Ridge(alpha=1.0)),
    'Polynomial(grau2)+Ridge': montar(Ridge(alpha=1.0), poly=True),
    'RandomForest': montar(RandomForestRegressor(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
}
list(modelos)

['LinearRegression', 'Ridge', 'Polynomial(grau2)+Ridge', 'RandomForest']

## Validacao cruzada (5-fold)

Metricas: R2 e RMSE medios no treino.

In [3]:
linhas = []
for nome, pipe in modelos.items():
    cv = cross_validate(pipe, X_train, y_train, cv=5,
                        scoring=['r2', 'neg_root_mean_squared_error'])
    linhas.append({
        'modelo': nome,
        'R2_medio': cv['test_r2'].mean(),
        'R2_std': cv['test_r2'].std(),
        'RMSE_medio': -cv['test_neg_root_mean_squared_error'].mean(),
    })
resultados = pd.DataFrame(linhas).sort_values('R2_medio', ascending=False)
resultados.round(4).reset_index(drop=True)

,modelo,R2_medio,R2_std,RMSE_medio
0,Polynomial(grau2)+Ridge,0.9656,0.0053,192.6913
1,RandomForest,0.9612,0.0040,205.0427
2,LinearRegression,0.9557,0.0043,219.2439
3,Ridge,0.9556,0.0045,219.5063


## Selecao e treino do melhor modelo

In [4]:
melhor_nome = resultados.iloc[0]['modelo']
melhor = modelos[melhor_nome]
melhor.fit(X_train, y_train)
print('Melhor modelo:', melhor_nome)

# Persiste pipeline completo + colunas de entrada esperadas
joblib.dump(melhor, MODELS_DIR / 'modelo_farmtech.pkl')
colunas = {
    'features': fe.FEATURES,
    'numeric': fe.NUMERIC_FEATURES,
    'categorical': fe.CATEGORICAL_FEATURES,
    'target': fe.TARGET,
    'melhor_modelo': melhor_nome,
}
with open(MODELS_DIR / 'model_columns.json', 'w', encoding='utf-8') as f:
    json.dump(colunas, f, ensure_ascii=False, indent=2)
print('Artefatos salvos em', MODELS_DIR)

Melhor modelo: Polynomial(grau2)+Ridge
Artefatos salvos em C:\Users\henri\Documents\ProjetosLocal\fase4-cap1\src\ml\models


## Justificativa

O modelo escolhido foi o de maior R2 medio na validacao cruzada, com RMSE
compativel. Modelos lineares servem de baseline; o termo polinomial e o
RandomForest capturam a nao-linearidade das faixas agronomicas. A avaliacao
detalhada no conjunto de teste esta em `03_avaliacao.ipynb`.